In [ ]:
import os
import sys

# 1. MONTAGGIO DRIVE E SETUP PERCORSI
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    DRIVE_ROOT = '/content/drive/MyDrive/AA-STAL/data_pipeline'

    if DRIVE_ROOT not in sys.path:
        sys.path.append(DRIVE_ROOT)

    print(f"Directory impostata su: {os.getcwd()}")
except ImportError:
    # Fallback per esecuzione in locale
    DRIVE_ROOT = '.'
    print("Esecuzione in locale")

# 2. INSTALLAZIONI DI SISTEMA (Usiamo ! bash commands per massima stabilità)
print("\n--- Inizio Installazione Transformers e Qwen ---")
!pip install -U git+https://github.com/huggingface/transformers.git
!pip install qwen-vl-utils accelerate

!pip install "bitsandbytes>=0.46.1"

print("\n--- Utility Pipeline e Fix OpenCV ---")
!pip install "opencv-python==4.10.0.84" "opencv-contrib-python==4.10.0.84" "opencv-python-headless==4.10.0.84"

print("\n=======================================================")
print("INSTALLAZIONE COMPLETATA. RIAVVIO IL KERNEL AUTOMATICAMENTE.")
print("Attendi 5 secondi, poi esegui la cella con il tuo script Python.")
print("=======================================================")

# 3. RIAVVIO AUTOMATICO DEL RUNTIME
import IPython
IPython.Application.instance().kernel.do_shutdown(True)


In [ ]:
import sys
import os

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    DRIVE_ROOT = '/content/drive/MyDrive/AA-STAL/data_pipeline'

    if DRIVE_ROOT not in sys.path:
        sys.path.append(DRIVE_ROOT)

    print(f"Directory impostata su: {os.getcwd()}")
except ImportError:
    # Fallback per esecuzione in locale
    DRIVE_ROOT = '.'
    print("Esecuzione in locale")

# Cambia la directory di lavoro in modo che i path relativi (es. configs/, saved_models/) funzionino
os.chdir(DRIVE_ROOT)
print(f"Directory di lavoro ripristinata su: {os.getcwd()}")

In [ ]:
import os
import json
import glob
import torch
import gc
import numpy as np
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

# ==========================================
# CONFIGURAZIONE E PARAMETRI
# ==========================================
PATH_DATA_ROOT = '/content/drive/MyDrive/AA-STAL/data_pipeline/DATA_ROOT'
PATH_VIDEOS_CROP_DECODE = os.path.join(PATH_DATA_ROOT, "Videos_crop_decode")
PATH_OBJ_DET_FINISHED = os.path.join(PATH_DATA_ROOT, "video_general_obj_det_finished")
PATH_OUTPUT_DATASET = os.path.join(PATH_DATA_ROOT, "action_recognition_finished")

AZIONI_CONSENTITE = [ "lift", "carry", "push", "pull", "pass", "lean_on", "touch", "hit", "play(instrument)", "grab",
                      "release", "press", "use", "throw", "clean", "knock", "squeeze", "cut", "open", "close", "watch",
                      "hold", "speak_to", "ride", "hug", "hold_hand_of", "hit", "bite", "caress", "pat", "wave", "point_to",
                      "chase", "feed", "kiss", "kick", "smell", "wave_hand_to", "lick", "drive", "shout_at", "get_on", "get_off",
                      "shake_hand_with" ]

WINDOW_SIZE = 30
FRAMES_TO_SAMPLE = 10

ORIGINAL_VIDEO_FPS = 30.0 
EFFECTIVE_FPS = FRAMES_TO_SAMPLE / (WINDOW_SIZE / ORIGINAL_VIDEO_FPS)

os.makedirs(PATH_OUTPUT_DATASET, exist_ok=True)

# ==========================================
# INIZIALIZZAZIONE MODELLO (Ottimizzato per T4)
# ==========================================
print("Caricamento del modello Qwen2.5-VL-7B-Instruct in 4-bit con SDPA...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# OTTIMIZZAZIONE MEMORIA 1: torch_dtype esplicito e attn_implementation="sdpa"
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa"
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")

# ==========================================
# FUNZIONI UTILI
# ==========================================
def scale_to_1000(bbox):
    if bbox is None:
        return None
    return [int(coord * 1000) for coord in bbox]

def clean_json_output(output_text):
    try:
        text = output_text.strip()
        if "```json" in text:
            text = text.split("```json")[1].split("```")[0].strip()
        elif "```" in text:
            text = text.split("```")[1].split("```")[0].strip()
        return json.loads(text)
    except Exception as e:
        print(f" Errore nel parsing del JSON generato dal modello: {e}")
        return {"actions": ["unknown", "unknown", "unknown"]}

def process_window(frames_paths, target_person_id, person_bbox, context_objects, action_list, video_fps):
    """Invia la finestra di frame e i metadati a Qwen per l'Action Recognition."""

    # 1. Costruzione contesto testuale (identica a prima)
    obj_context_str = ""
    if context_objects:
        obj_context_str = "Objects detected in the scene with potential interactions:\n"
        for obj_name, obj_box in context_objects.items():
            scaled_obj_box = scale_to_1000(obj_box)
            obj_context_str += f"- {obj_name} located at: <box>({scaled_obj_box[0]},{scaled_obj_box[1]},{scaled_obj_box[2]},{scaled_obj_box[3]})</box>\n"
    else:
        obj_context_str = "No relevant objects detected around the subject.\n"

    scaled_person_box = scale_to_1000(person_bbox)
    person_box_str = f"<box>({scaled_person_box[0]},{scaled_person_box[1]},{scaled_person_box[2]},{scaled_person_box[3]})</box>"

    prompt_text = (
        f"Analyze the provided video sequence.\n"
        f"It tracks the actor '{target_person_id}'. "
        f"The target is exactly at the coordinates: {person_box_str}.\n\n"
        f"{obj_context_str}\n"
        f"Based on the actor '{target_person_id}' movements, select the 3 most likely actions, ordered from highest to lowest probability, EXCLUSIVELY from the following options:\n"
        f"{json.dumps(action_list, ensure_ascii=False)}\n\n"
        f"Answer STRICTLY and ONLY with a valid JSON object in the format:\n"
        f'{{"actions": ["MOST_LIKELY_ACTION", "SECOND_POSSIBLE_ACTION"]}}\n'
    )

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "video",
                    "video": frames_paths,
                    "max_pixels": 420 * 420,
                    "fps": video_fps
                },
                {
                    "type": "text",
                    "text": prompt_text
                }
            ]
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")

    # 4. Inferenza protetta
    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=128, do_sample=False)
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]

    # 5. Distruzione aggressiva
    for k in list(inputs.keys()):
        del inputs[k]
    del inputs
    if video_inputs is not None:
        del video_inputs
    del generated_ids
    del generated_ids_trimmed

    return clean_json_output(output_text)

# ==========================================
# LOOP PRINCIPALE DI ELABORAZIONE
# ==========================================
video_folders = glob.glob(os.path.join(PATH_OBJ_DET_FINISHED, "*"))

for video_folder in video_folders:
    video_name = os.path.basename(video_folder)
    print(f"\n Elaborazione Video: {video_name}")

    json_path = os.path.join(video_folder, "*.json")
    json_files = glob.glob(json_path)

    if not json_files:
        print(f" [Skip] Nessun file JSON di tracking trovato per {video_name}")
        continue

    with open(json_files[0], 'r') as f:
        tracking_data = json.load(f)

    frames_dir = os.path.join(PATH_VIDEOS_CROP_DECODE, video_name)
    all_frames = sorted(glob.glob(os.path.join(frames_dir, "*.jpg")))

    total_frames = len(all_frames)
    if total_frames == 0:
        print(f" [Skip] Nessun frame trovato nella cartella dei video grezzi per {video_name}")
        continue

    detected_objects = tracking_data.get("detected_objects", {})
    people_ids = [k for k in detected_objects.keys() if k.startswith("person_")]

    if not people_ids:
        print(f" Nessun attore umano trovato nel JSON di {video_name}")
        continue

    for person_id in people_ids:
        print(f"  Analisi azioni per il soggetto: {person_id}")
        person_bboxes = detected_objects[person_id].get("bbox", [])

        person_action_log = {}

        for start_idx in range(0, total_frames, WINDOW_SIZE):
            end_idx = min(start_idx + WINDOW_SIZE, total_frames)
            window_len = end_idx - start_idx

            mid_idx = start_idx + (window_len // 2)

            if mid_idx >= len(person_bboxes) or person_bboxes[mid_idx] is None:
                continue

            current_person_bbox = person_bboxes[mid_idx]

            context_objects = {}
            for obj_key, obj_data in detected_objects.items():
                if obj_key == person_id:
                    continue
                obj_boxes = obj_data.get("bbox", [])
                if mid_idx < len(obj_boxes) and obj_boxes[mid_idx] is not None:
                    context_objects[obj_key] = obj_boxes[mid_idx]

            window_frames = all_frames[start_idx:end_idx]
            sample_indices = np.linspace(0, len(window_frames) - 1, min(FRAMES_TO_SAMPLE, len(window_frames)), dtype=int)
            sampled_frames_paths = [window_frames[i] for i in sample_indices]

            result_json = process_window(
                frames_paths=sampled_frames_paths,
                target_person_id=person_id,
                person_bbox=current_person_bbox,
                context_objects=context_objects,
                action_list=AZIONI_CONSENTITE,
                video_fps=EFFECTIVE_FPS
            )

            gc.collect()
            torch.cuda.empty_cache()

            timestamp_key = f"frames_{start_idx}_to_{end_idx}"
            # Salva l'intero array delle top-3 azioni nel log
            person_action_log[timestamp_key] = result_json.get("actions", ["unknown", "unknown", "unknown"])
            print(f"    -> Finestra {start_idx}-{end_idx}: {person_action_log[timestamp_key]}")

        output_filename = f"{video_name}_{person_id}_actions.json"
        output_file_path = os.path.join(PATH_OUTPUT_DATASET, output_filename)

        with open(output_file_path, 'w', encoding='utf-8') as out_f:
            json.dump(person_action_log, out_f, indent=4, ensure_ascii=False)

        print(f"   Salvataggio completato: {output_file_path}")

        # Pulisci la memoria al termine di ogni soggetto
        gc.collect()
        torch.cuda.empty_cache()

print("\n Pipeline completata con successo su tutti i video.")